# Тестовое задание: Анализ пролонгаций аккаунт-менеджеров
**Цель:** Рассчитать коэффициенты пролонгации договоров за 2023 год для каждого менеджера и отдела в целом на основе предоставленных выгрузок.

In [ ]:
!pip install xlsxwriter

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### Шаг 1. Загрузка и очистка данных
Загружаем данные и приводим финансовые показатели к числовому формату. Текстовые статусы оплат приравниваем к нулю. Перед запуском убедитесь, что файлы `financial_data.csv` и `prolongations.csv` загружены в левую панель Colab.

In [ ]:
prolongations = pd.read_csv('prolongations.csv')
fin_data = pd.read_csv('financial_data.csv')

prolongations['month'] = prolongations['month'].str.capitalize()

def clean_currency(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, str):
        x = x.replace('\xa0', '').replace(' ', '').replace(',', '.')
        x = x.lower().strip()
        if x in ['стоп', 'в ноль', 'end']:
            return 0.0
        try:
            return float(x)
        except ValueError:
            return 0.0
    return float(x)

month_columns = fin_data.columns[2:-1]
for col in month_columns:
    fin_data[col] = fin_data[col].apply(clean_currency)

fin_data_grouped = fin_data.groupby('id')[month_columns].sum().reset_index()
df = fin_data_grouped.merge(prolongations[['id', 'month', 'AM']], on='id', how='left')

### Шаг 2. Реализация бизнес-логики (Расчет коэффициентов)
Рассчитываем базовые метрики по каждому месяцу 2023 года.

In [ ]:
target_months = [
    'Январь 2023', 'Февраль 2023', 'Март 2023', 'Апрель 2023', 
    'Май 2023', 'Июнь 2023', 'Июль 2023', 'Август 2023', 
    'Сентябрь 2023', 'Октябрь 2023', 'Ноябрь 2023', 'Декабрь 2023'
]

all_months = list(month_columns)
month_to_idx = {m: i for i, m in enumerate(all_months)}
results = []

for t_month in target_months:
    t_idx = month_to_idx[t_month]
    m1_month = all_months[t_idx - 1]
    m2_month = all_months[t_idx - 2]
    
    mask_c1 = df['month'] == m1_month
    df_c1 = df[mask_c1]
    base_c1 = df_c1.groupby('AM')[m1_month].sum().rename('base_1')
    fact_c1 = df_c1.groupby('AM')[t_month].sum().rename('fact_1')
    
    mask_c2 = (df['month'] == m2_month) & (df[m1_month] == 0)
    df_c2 = df[mask_c2]
    base_c2 = df_c2.groupby('AM')[m2_month].sum().rename('base_2')
    fact_c2 = df_c2.groupby('AM')[t_month].sum().rename('fact_2')
    
    month_stats = pd.concat([base_c1, fact_c1, base_c2, fact_c2], axis=1).fillna(0)
    month_stats['Месяц'] = t_month
    month_stats = month_stats.reset_index()
    results.append(month_stats)

final_df = pd.concat(results, ignore_index=True)

### Шаг 3. Подготовка аналитических срезов
Формируем агрегированные таблицы для итогового отчета с явным указанием порядка столбцов для корректного экспорта.

In [ ]:
dept_monthly = final_df.groupby('Месяц').sum(numeric_only=True).reset_index()
dept_monthly['Коэффициент 1'] = np.where(dept_monthly['base_1'] > 0, dept_monthly['fact_1'] / dept_monthly['base_1'], 0)
dept_monthly['Коэффициент 2'] = np.where(dept_monthly['base_2'] > 0, dept_monthly['fact_2'] / dept_monthly['base_2'], 0)
dept_monthly_final = dept_monthly[['Месяц', 'base_1', 'fact_1', 'Коэффициент 1', 'base_2', 'fact_2', 'Коэффициент 2']]
dept_monthly_final['Месяц'] = pd.Categorical(dept_monthly_final['Месяц'], categories=target_months, ordered=True)
dept_monthly_final = dept_monthly_final.sort_values('Месяц')

mgr_yearly = final_df.groupby('AM').sum(numeric_only=True).reset_index()
mgr_yearly['Коэффициент 1'] = np.where(mgr_yearly['base_1'] > 0, mgr_yearly['fact_1'] / mgr_yearly['base_1'], 0)
mgr_yearly['Коэффициент 2'] = np.where(mgr_yearly['base_2'] > 0, mgr_yearly['fact_2'] / mgr_yearly['base_2'], 0)
mgr_yearly_final = mgr_yearly[['AM', 'base_1', 'fact_1', 'Коэффициент 1', 'base_2', 'fact_2', 'Коэффициент 2']]

final_df['coef_1'] = np.where(final_df['base_1'] > 0, final_df['fact_1'] / final_df['base_1'], 0)
pivot_c1 = final_df.pivot(index='AM', columns='Месяц', values='coef_1').reindex(columns=target_months).fillna(0)

final_df['coef_2'] = np.where(final_df['base_2'] > 0, final_df['fact_2'] / final_df['base_2'], 0)
pivot_c2 = final_df.pivot(index='AM', columns='Месяц', values='coef_2').reindex(columns=target_months).fillna(0)

### Шаг 4. Экспорт в Excel
Сохраняем результаты в файл с применением строгого форматирования и точной адресацией ячеек.

In [ ]:
with pd.ExcelWriter('Analytical_Report_Prolongation.xlsx', engine='xlsxwriter') as writer:
    workbook = writer.book
    header_fmt = workbook.add_format({'bold': True, 'align': 'center', 'valign': 'vcenter', 'bg_color': '#FFF2CC', 'border': 1})
    pct_fmt = workbook.add_format({'num_format': '0.00%', 'border': 1})
    money_fmt = workbook.add_format({'num_format': '#,##0.00', 'border': 1})
    border_fmt = workbook.add_format({'border': 1})
    
    ws1 = workbook.add_worksheet('Весь отдел')
    ws1.merge_range('B1:D1', 'Пролонгации в первый месяц', header_fmt)
    ws1.merge_range('E1:G1', 'Пролонгации через месяц', header_fmt)
    headers = ['Месяц', 'к пролонгации', 'пролонгировано', 'Коэффициент', 'к пролонгации', 'пролонгировано', 'Коэффициент']
    for col_num, value in enumerate(headers):
        ws1.write(1, col_num, value, header_fmt)
    for row_num, row_data in enumerate(dept_monthly_final.values, 2):
        ws1.write(row_num, 0, row_data[0], border_fmt)
        ws1.write(row_num, 1, row_data[1], money_fmt)
        ws1.write(row_num, 2, row_data[2], money_fmt)
        ws1.write(row_num, 3, row_data[3], pct_fmt)
        ws1.write(row_num, 4, row_data[4], money_fmt)
        ws1.write(row_num, 5, row_data[5], money_fmt)
        ws1.write(row_num, 6, row_data[6], pct_fmt)
    ws1.set_column('A:G', 18)

    ws2 = workbook.add_worksheet('Менеджеры за год')
    ws2.merge_range('B1:D1', 'Пролонгации в первый месяц', header_fmt)
    ws2.merge_range('E1:G1', 'Пролонгации через месяц', header_fmt)
    for col_num, value in enumerate(['Менеджер', 'к пролонгации', 'пролонгировано', 'Коэффициент', 'к пролонгации', 'пролонгировано', 'Коэффициент']):
        ws2.write(1, col_num, value, header_fmt)
    for row_num, row_data in enumerate(mgr_yearly_final.values, 2):
        ws2.write(row_num, 0, row_data[0], border_fmt)
        ws2.write(row_num, 1, row_data[1], money_fmt)
        ws2.write(row_num, 2, row_data[2], money_fmt)
        ws2.write(row_num, 3, row_data[3], pct_fmt)
        ws2.write(row_num, 4, row_data[4], money_fmt)
        ws2.write(row_num, 5, row_data[5], money_fmt)
        ws2.write(row_num, 6, row_data[6], pct_fmt)
    ws2.set_column('A:A', 35)
    ws2.set_column('B:G', 18)

    ws3 = workbook.add_worksheet('Менеджеры по месяцам')
    ws3.merge_range('A1:B1', 'Коэффициент 1', header_fmt)
    ws3.merge_range('C1:N1', 'Месяц', header_fmt)
    ws3.write(1, 0, 'Менеджер', header_fmt)
    ws3.write(1, 1, '', header_fmt)
    for col_num, value in enumerate(target_months, 2):
        ws3.write(1, col_num, value, header_fmt)
    row_offset = 2
    for am in pivot_c1.index:
        ws3.merge_range(row_offset, 0, row_offset, 1, am, border_fmt)
        for col_num, val in enumerate(pivot_c1.loc[am].values, 2):
            ws3.write(row_offset, col_num, val, pct_fmt)
        row_offset += 1
        
    row_offset += 2
    ws3.merge_range(f'A{row_offset}:B{row_offset}', 'Коэффициент 2', header_fmt)
    ws3.merge_range(f'C{row_offset}:N{row_offset}', 'Месяц', header_fmt)
    ws3.write(row_offset, 0, 'Менеджер', header_fmt)
    ws3.write(row_offset, 1, '', header_fmt)
    for col_num, value in enumerate(target_months, 2):
        ws3.write(row_offset, col_num, value, header_fmt)
    row_offset += 1
    for am in pivot_c2.index:
        ws3.merge_range(row_offset, 0, row_offset, 1, am, border_fmt)
        for col_num, val in enumerate(pivot_c2.loc[am].values, 2):
            ws3.write(row_offset, col_num, val, pct_fmt)
        row_offset += 1
    ws3.set_column('A:B', 20)
    ws3.set_column('C:N', 15)
print('Аналитический отчет успешно сформирован!')